# Reproduce: automated eight cardiovascular diameters on chest CT

Guided Colab reproduction of the **measurement pipeline mechanics** on a public, license-clean chest CT,
using immutable release tag `v1.0.2` and the published model-weight SHA256 pins.

**What this shows:** that the released pipeline runs end-to-end from a raw CT NIfTI to the eight
diameters (PT, RPA, LPA, AA, RA, RV, LA, LV), and that the weights match the published run.

**What this is NOT:** an accuracy or clinical evaluation. This notebook only reproduces the
computational mechanics on one public sample; see the manuscript for the reader-referenced
agreement analysis and its limitations.

**You need:** a Colab GPU runtime (Runtime → Change runtime type → GPU) and your **own** free
TotalSegmentator academic license (https://backend.totalsegmentator.com/license-academic/). The weights
are **not** redistributed here — you download them under your own license and this notebook verifies
their SHA256 against the published pins. `totalseg_set_license` stores the key in
`~/.totalsegmentator/config.json` inside this ephemeral runtime; delete the runtime when finished.

Runtime: ~3–5 min on a T4.

## 1 · Get the frozen code and install dependencies

In [ ]:
REPO_URL = "https://github.com/honeia85/cardiovascular-diameter-measurement-ct"
REF = "v1.0.2"  # immutable release tag
PIPE = "/content/pipeline_repo"

import os, shutil, subprocess, sys
assert sys.version_info[:2] == (3, 12), f"Python 3.12.x required; got {sys.version}"
if os.path.exists(PIPE):
    shutil.rmtree(PIPE)
subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REF, REPO_URL, PIPE], check=True)
actual_ref = subprocess.check_output(["git", "-C", PIPE, "describe", "--tags", "--exact-match"], text=True).strip()
assert actual_ref == REF, (actual_ref, REF)
commit = subprocess.check_output(["git", "-C", PIPE, "rev-parse", "HEAD"], text=True).strip()

# The published measurements used torch 2.5.1+cu118/CUDA 11.8. A clean Colab T4
# smoke test used the compatible 2.5.1+cu124 build; do not force cu118 onto Colab.
def installed_torch_version():
    try:
        return subprocess.check_output([sys.executable, "-c", "import torch; print(torch.__version__)"], text=True).strip()
    except subprocess.CalledProcessError:
        return ""

torch_before = installed_torch_version()
if torch_before.split("+")[0] != "2.5.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.5.1", "torchvision==0.20.1",
                    "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PIPE}/requirements.txt"], check=True)

import torch
assert torch.__version__.split("+")[0] == "2.5.1", torch.__version__
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"
print("release", REF, "| commit", commit[:12], "| torch", torch.__version__,
      "| CUDA runtime", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0))

## 2 · Set YOUR TotalSegmentator license, download and VERIFY the weights
The hidden prompt avoids echoing the key, but `totalseg_set_license` writes it to
`~/.totalsegmentator/config.json` in this ephemeral runtime. It is not committed or copied into outputs;
delete the runtime when finished. Final `verify_weights.py` execution confirms both checkpoints.

In [ ]:
import getpass, subprocess, sys
from pathlib import Path
LICENSE = getpass.getpass("TotalSegmentator academic license (hidden; stored only in this runtime): ").strip()
subprocess.run(["totalseg_set_license", "-l", LICENSE], check=True)
del LICENSE  # remove the in-memory copy; TotalSegmentator retains its runtime config file
license_config = Path.home() / ".totalsegmentator" / "config.json"
assert license_config.exists(), f"license config not created: {license_config}"
print("license registered in ephemeral runtime:", license_config)
# heartchambers_highres (Dataset301); the free total_3mm (Dataset297) ROI model auto-downloads on first run
subprocess.run(["totalseg_download_weights", "-t", "heartchambers_highres"], check=True)
print("Dataset301 downloaded; Dataset297 downloads on the first pipeline run.")

## 3 · Fetch a public, license-clean sample CT
Default: the fixed public LungCT-Diagnosis R_020 series (CC BY 3.0) is retrieved through the NBIA API
and converted with `dcm2niix`. You may instead set `DIRECT_NIFTI_URL` to another public `.nii.gz`.
The volume is explicitly reoriented and checked for an axis-aligned LPS+ grid before use. The helper
only permutes/flips axes; oblique or sheared inputs require separate resampling. Never use clinical or
PHI-bearing data in this demonstration.

In [ ]:
import glob, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path
import nibabel as nib
import numpy as np

DIRECT_NIFTI_URL = ""  # optional public .nii.gz; leave empty for the fixed TCIA sample
SERIES_UID = "1.3.6.1.4.1.14519.5.2.1.4320.5030.261781697101054597239292572759"  # R_020
BASE = "https://services.cancerimagingarchive.net/nbia-api/services/v1"

if DIRECT_NIFTI_URL:
    urllib.request.urlretrieve(DIRECT_NIFTI_URL, "/content/sample_ct_raw.nii.gz")
    raw_ct = "/content/sample_ct_raw.nii.gz"
else:
    subprocess.run(["apt-get", "-qq", "install", "-y", "dcm2niix"],
                   check=True, capture_output=True, text=True)
    print("fixed public series:", SERIES_UID, "(LungCT-Diagnosis R_020, CC BY 3.0)")
    urllib.request.urlretrieve(f"{BASE}/getImage?SeriesInstanceUID={SERIES_UID}", "/content/series.zip")
    if os.path.exists("/content/dcm"):
        shutil.rmtree("/content/dcm")
    os.makedirs("/content/dcm", exist_ok=True)
    zipfile.ZipFile("/content/series.zip").extractall("/content/dcm")
    subprocess.run(["dcm2niix", "-z", "y", "-o", "/content", "-f", "sample_ct_raw", "/content/dcm"],
                   check=True, capture_output=True, text=True)
    converted = glob.glob("/content/sample_ct_raw*.nii.gz")
    assert converted, "dcm2niix produced no NIfTI"
    raw_ct = max(converted, key=os.path.getsize)

raw_img = nib.load(raw_ct)
assert len(raw_img.shape) == 3, f"expected 3D CT, got {raw_img.shape}"
source_codes = nib.aff2axcodes(raw_img.affine)
CT = "/content/sample_ct_lps.nii.gz"
Path(CT).unlink(missing_ok=True)  # helper intentionally refuses to overwrite
subprocess.run([sys.executable, f"{PIPE}/reorient_nifti_lps.py", raw_ct, CT], check=True)

check_img = nib.load(CT)
target_codes = nib.aff2axcodes(check_img.affine)
zooms = np.asarray(check_img.header.get_zooms()[:3], dtype=float)
linear = np.asarray(check_img.affine[:3, :3], dtype=float)
assert target_codes == ("L", "P", "S"), target_codes
assert np.all(np.isfinite(check_img.affine)) and np.all(np.isfinite(zooms))
assert np.all(zooms > 0) and abs(np.linalg.det(linear)) > 1e-8
axis_lengths = np.linalg.norm(linear, axis=0)
assert np.allclose(axis_lengths, zooms, rtol=1e-5, atol=1e-6)
directions = linear / axis_lengths
assert np.allclose(directions, np.diag([-1.0, -1.0, 1.0]), rtol=0.0, atol=1e-5), \
    "oblique/sheared affine: resample separately to an axis-aligned LPS+ grid"
assert np.isclose(zooms[0], zooms[1], rtol=1e-4, atol=1e-4), f"anisotropic in-plane spacing: {zooms}"
print("CT ready:", CT, "| shape", check_img.shape, "| axes", source_codes, "→", target_codes,
      "| spacing mm", tuple(float(x) for x in zooms))

## 4 · Run the frozen pipeline and show the eight diameters

In [ ]:
import subprocess, sys, glob, json, shutil, os
if os.path.exists("/content/out"):
    shutil.rmtree("/content/out")
env = dict(os.environ); env["TOTALSEG_EXE"] = shutil.which("TotalSegmentator") or "TotalSegmentator"
r = subprocess.run([sys.executable, f"{PIPE}/run_pipeline.py", "-i", CT, "-o", "/content/out", "-d", "gpu:0"],
                   check=True, capture_output=True, text=True, env=env)
print(r.stdout[-800:])
print(r.stderr[-800:])

mj = glob.glob("/content/out/**/*_measurements.json", recursive=True)[0]
d = json.load(open(mj))
order = ["PT", "RPA", "LPA", "AA", "RA", "RV", "LA", "LV"]
assert len(d.get("measurements", {})) == 8 and d.get("missing") == [], d.get("missing")
print(f"\n{len(d['measurements'])}/8 structures | missing: {d['missing']} | {d['elapsed_sec']} s\n")
for s in order:
    v = d["measurements"].get(s, {})
    print(f"  {s:4s} {v.get('diameter_mm', float('nan')):6.2f} mm")
ratios = d.get("ratios", {})
for key in ("PT_AA_ratio", "RV_LV_ratio"):
    assert key in ratios, f"missing ratio output: {key}"
    assert ratios[key] is not None, f"null ratio output: {key}"
    print(f"  {key:12s} {ratios[key]}")

In [ ]:
# Re-verify weights now that the free ROI model (Dataset297) has also been fetched
import subprocess, sys
subprocess.run([sys.executable, f"{PIPE}/verify_weights.py"], check=True)

## 5 · Interpreting the output

If you see `8/8 structures`, `missing: []`, and `verify_weights.py` reporting that all weights match the
published run, the released pipeline has reproduced end-to-end on your machine with the exact published
checkpoints.

The diameter values are for one public sample and are **not** compared to any reference here. See the
manuscript for the reader-referenced agreement findings and limitations.

Constraints honored by this notebook: the license is saved only in the ephemeral runtime's
`~/.totalsegmentator/config.json`; segmentation weights are downloaded under your own license and are
not redistributed; only the fixed public, license-clean sample is used. Delete the runtime when done.